In [ ]:
# 02_recourse_generation.ipynb
# Generate counterfactual recourse for rejected members under three governance
# scenarios (A: unconstrained, B: immutability enforced, C: full guardrail with a
# +/-20% action horizon), for the sampled group comparison and for the entire
# disadvantaged rejected pool. Nominal features are declared categorical to the
# generator so the search ranges over valid feature levels. This is the longest
# stage; results are cached. No identifying paths are printed.

import time, pickle, logging
from pathlib import Path
import numpy as np, pandas as pd
import dice_ml
from dice_ml import Dice

ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
CACHE = ROOT / "results" / "cache"
RANDOM_STATE = 42
ACTION_HORIZON = 0.20
RECOURSE_N = 200

with open(CACHE / "nb1_artifacts.pkl", "rb") as f:
    A = pickle.load(f)
rf = A["rf"]; X_train = A["X_train"]; y_train = A["y_train"]; X_test = A["X_test"]
FEATURES = A["FEATURES"]; IMMUTABLE = A["IMMUTABLE"]; ACTIONABLE = A["ACTIONABLE"]; TARGET = A["TARGET"]
group_test = A["group_test"]

logging.getLogger("dice_ml").setLevel(logging.ERROR)
try:
    import tqdm; from functools import partialmethod
    tqdm.tqdm.__init__ = partialmethod(tqdm.tqdm.__init__, disable=True)
except Exception:
    pass

# Declare which features are continuous vs categorical for the generator.
CONTINUOUS = ["mn_credit_tenure", "amt_credit_r3m", "amt_usage_r3m", "cnt_merchant_r3m",
              "amt_shop", "amt_eat", "amt_leisure", "amt_credit_limit", "age"]
train_for_dice = X_train.copy(); train_for_dice[TARGET] = y_train.values
d = dice_ml.Data(dataframe=train_for_dice, continuous_features=CONTINUOUS, outcome_name=TARGET)
m = dice_ml.Model(model=rf, backend="sklearn")
exp = Dice(d, m, method="random")

def per_feature_ranges(row):
    rng = {}
    for f in ACTIONABLE:
        v = float(row[f]); lo, hi = sorted([v * (1 - ACTION_HORIZON), v * (1 + ACTION_HORIZON)]); rng[f] = [lo, hi]
    return rng

def gen_recourse(subset, vary, use_range):
    results = []
    for _, row in subset.iterrows():
        query = row[FEATURES].to_frame().T
        try:
            kw = dict(total_CFs=4, desired_class=1, features_to_vary=vary, verbose=False)
            if use_range: kw["permitted_range"] = per_feature_ranges(row)
            cf = exp.generate_counterfactuals(query, **kw)
            cfdf = cf.cf_examples_list[0].final_cfs_df
            results.append(None if cfdf is None else cfdf[FEATURES].reset_index(drop=True))
        except Exception:
            results.append(None)
    return results

# Rejected members eligible for recourse.
test_df = X_test.copy(); test_df["group"] = group_test.values; test_df["pred"] = rf.predict(X_test[FEATURES])
rejected = test_df[(test_df["pred"] == 0) & (test_df["group"].isin(["advantaged", "disadvantaged"]))].copy()
vary_A = FEATURES; vary_B = [f for f in FEATURES if f not in IMMUTABLE]; vary_C = ACTIONABLE

# Sampled group comparison (n=200 per group).
recourse = {}
for gname in ["disadvantaged", "advantaged"]:
    sub = rejected[rejected.group == gname]
    s = sub.sample(n=min(RECOURSE_N, len(sub)), random_state=RANDOM_STATE)
    recourse[gname] = {"A": gen_recourse(s, vary_A, False), "B": gen_recourse(s, vary_B, False),
                       "C": gen_recourse(s, vary_C, True), "_sample": s}
with open(CACHE / "recourse_main.pkl", "wb") as f:
    pickle.dump(recourse, f)

# Full disadvantaged rejected pool (measured, not extrapolated).
dis_full = rejected[rejected.group == "disadvantaged"].copy()
recourse_dis_full = {scen: gen_recourse(dis_full, vary, use_range)
                     for scen, (vary, use_range) in {"A": (vary_A, False), "B": (vary_B, False), "C": (vary_C, True)}.items()}
with open(CACHE / "recourse_dis_full.pkl", "wb") as f:
    pickle.dump(recourse_dis_full, f)
with open(CACHE / "nb2_frames.pkl", "wb") as f:
    pickle.dump({"rejected": rejected, "dis_full": dis_full}, f)

print("Notebook 02 complete. Recourse generated for the sampled comparison and the full disadvantaged pool.")
